In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

### 🧠 What is `SummarizationMiddleware`?

As an agent runs for many turns, `messages[]` keeps growing. Eventually it blows past the LLM's context window or gets expensive. You could just truncate old messages, but then you lose important context.

`SummarizationMiddleware` is the middle ground:

1. ⏱️ Watches the message count (or token count)
2. 📝 When it crosses a threshold, asks the LLM to **summarize** the older messages into one condensed message
3. 🔄 Replaces those old messages with the summary
4. ✅ Keeps recent messages intact

So instead of:
```
[msg1, msg2, msg3, msg4, msg5, msg6, msg7, msg8]  ← growing forever
```

You get:
```
[summary_of_1_through_5, msg6, msg7, msg8]  ← bounded size
```

The agent still "remembers" what happened in the early turns (via the summary), but the actual token count stays manageable.

> 💡 This is essentially **automatic RAG over your own conversation history**.

**Key parameters below:**
- `model="gpt-4o-mini"` — a cheap, fast model used just for summarization (not the main agent model)
- `trigger=("tokens", 100)` — summarize when messages exceed 100 tokens
- `keep=("messages", 1)` — always keep the most recent 1 message untouched

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is asking about various details regarding Lunapolis, a fictional city on the moon, including its capital, weather, population of cheese miners, and potential union activities.\n\n## SUMMARY\n- The capital of the moon is identified as Lunapolis.\n- The weather in Lunapolis is described as clear skies, with temperatures ranging from a high of 120C to a low of -100C.\n- Lunapolis is home to 100,000 cheese miners.\n- There is speculation that the cheese miners' union may strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='6b927c78-d20b-4eec-b866-60f9cbf5d0db'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='e6bc9308-d78e-419e-9e62

In [ ]:
print(response["messages"][0].content)

## Trim/delete messages

In [4]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [5]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [6]:
response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="My device won't turn on. What should I do?"),
            ToolMessage(
                content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"
            ),
            AIMessage(content="Is the device plugged in and turned on?"),
            HumanMessage(content="Yes, it's plugged in and turned on."),
            ToolMessage(
                content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"
            ),
            AIMessage(content="Is the device showing any lights or indicators?"),
            HumanMessage(content="What's the temperature of the device?"),
        ]
    },
    {"configurable": {"thread_id": "2"}},
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='5a559ee9-a4c7-4203-bca6-b782fef85e40'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='74551317-3878-48e4-96ec-b401b5dc3f2e', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='6a344bfd-8d7a-4f4f-851d-4ea4d0ec576a'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='0e3579d1-ab92-4f4c-a677-d8c4c8b0b654', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='1697ddf0-a447-4a92-b3d6-2cfa1962f2ae'),
              AIMessage(content='I can’t read the device’s temperature from here. If you want t

In [7]:
print(response["messages"][-1].content)

I can’t read the device’s temperature from here. If you want to check manually, try these:

- Feel for heat: is the back/bottom or charger area hot to the touch? If yes, it may be overheating.
- If it’s hot: power it off, unplug it, and let it cool on a hard surface with good ventilation for 15–30 minutes.
- Check ventilation: make sure vents aren’t blocked by dust, fabric, or furniture. 
- Inspect for dust and debris and consider cleaning the vents with compressed air (while powered off and unplugged).
- After it’s cooled, try powering it on again. If it still won’t start or repeatedly overheats, try a different charger/outlet.
- If you notice swelling, burning smell, or it gets extremely hot quickly, stop using it and contact support.

If you can share the device type (brand/model) and OS, I can give more specific steps to check temperatures and diagnose boot issues.
